# Advanced Python Rich Comparisons — 18 Problems with Complete Solutions

**Topic:** Python's `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, reflected comparison dispatch, `NotImplemented`, `functools.total_ordering`, ordering invariants, sorting, and hashing. This is an expanded problem set grounded in the supplied *Rich Comparisons* lesson, which demonstrates vectors, tuples, reflected operators and `total_ordering`.

**Audience:** Intermediate to advanced Python learners. **Runtime:** Python 3.10+; Python standard library only. **Format:** 18 challenges, executable model solutions, edge cases, and assertion-based tests. Run **Kernel → Restart & Run All**: the notebook is designed to execute top-to-bottom, without hidden state or network dependencies.

### Ground rules / corrections that matter

- `==` models *value equality*; `is` models *object identity*. Without a custom `__eq__`, ordinary user objects usually compare equal only to themselves; `object.__eq__` can return `NotImplemented` for different objects, and the overall equality operation then has identity-based fallback.
- Rich comparisons use **reflected pairs**: `<` ↔ `>`, `<=` ↔ `>=`, `==` ↔ `==`, and `!=` ↔ `!=`. Python does **not** automatically derive `<=` from `<` and `==`.
- Return the singleton `NotImplemented` (not `False`, and not `NotImplementedError`) for an **unsupported operand type**. When neither operand supports ordering, Python raises `TypeError`.
- `@total_ordering` derives missing methods from equality plus one ordering method, but cannot repair an inconsistent comparison contract. For an ordered type, equal ordering keys should normally imply equality unless you deliberately design a preorder and accept its consequences.
- If `a == b` is true and both objects are hashable, **their hashes must match**. Mutable value objects should normally be unhashable. Do not sort mutually incomparable objects without an explicit policy.

Every code cell either defines an independent example or checks a previously defined model. All deliberate error cases are caught and verified; an uncaught exception is a notebook test failure.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal
from functools import cmp_to_key, total_ordering
from itertools import count, product
from math import hypot, isfinite, isnan
import heapq
import re


def expect_raises(exception_type, operation):
    """Assert that a zero-argument operation raises the specified exception."""
    try:
        operation()
    except exception_type as exc:
        return str(exc)
    except Exception as exc:
        raise AssertionError(
            f"Expected {exception_type.__name__}, got {type(exc).__name__}"
        ) from exc
    raise AssertionError(f"Expected {exception_type.__name__}, but no exception occurred")

print("Standard-library setup complete. All exercises use assertions as tests.")

Standard-library setup complete. All exercises use assertions as tests.


## Problem 01 — Build a reflected-dispatch laboratory

**Challenge.** Create a class implementing *only* `__lt__` and `__eq__`. Demonstrate that `right > left` can invoke `left.__lt__(right)`, but `left <= right` is still unsupported. Record which methods are called, rather than assuming Python rewrites source code literally.

### Worked solution

In [2]:
class Probe:
    def __init__(self, value, log):
        self.value = value
        self.log = log

    def __lt__(self, other):
        self.log.append(("__lt__", self.value, getattr(other, "value", None)))
        if not isinstance(other, Probe):
            return NotImplemented
        return self.value < other.value

    def __eq__(self, other):
        self.log.append(("__eq__", self.value, getattr(other, "value", None)))
        if not isinstance(other, Probe):
            return NotImplemented
        return self.value == other.value


calls = []
small, large = Probe(1, calls), Probe(4, calls)
assert small < large
assert large > small           # Can be handled through small.__lt__(large).
assert small != large          # object.__ne__ delegates to equality for this class.
assert calls[:2] == [("__lt__", 1, 4), ("__lt__", 1, 4)]
assert "not supported" in expect_raises(TypeError, lambda: small <= large)
print("Dispatch trace:", calls)
print("<= is not synthesized automatically: confirmed")

Dispatch trace: [('__lt__', 1, 4), ('__lt__', 1, 4), ('__eq__', 1, 4)]
<= is not synthesized automatically: confirmed


**Why this works.** `>` can dispatch to the opposite operand's `<` implementation, including when both objects have the same type. `<=` requires `__le__` or a cooperating `__ge__`; Python will not infer it from `__eq__` plus `__lt__`.

**Further challenge (try before reading the next problem).** Add `__le__` and check the truth table for three values.

## Problem 02 — Distinguish `NotImplemented`, `False`, and exceptions

**Challenge.** Implement two unrelated types that cooperate for equality. The first type must decline an unknown operand instead of returning `False`; the second type then decides the result. Check unsupported ordering separately and prove `NotImplemented` is a return value, not an exception.

### Worked solution

In [3]:
class First:
    def __eq__(self, other):
        return NotImplemented


class Second:
    def __eq__(self, other):
        if isinstance(other, First):
            return True
        return NotImplemented


left, right = First(), Second()
assert First.__eq__(left, right) is NotImplemented
assert left == right
assert right == left
assert (left != right) is False

assert "not supported" in expect_raises(TypeError, lambda: left < right)

class Refuses:
    def __eq__(self, other):
        return False  # Bad design for multi-type cooperation: prevents fallback.

assert (Refuses() == right) is False
print("Equality fallback works; unsupported order correctly raises TypeError.")

Equality fallback works; unsupported order correctly raises TypeError.


**Why this works.** Python tries the other operand's equality method when the first returns `NotImplemented`; an early `False` prevents that cooperation. `NotImplementedError` is an exception for abstract or unfinished features, not the sentinel for unsupported rich-comparison operands.

**Further challenge (try before reading the next problem).** Instrument both classes to record the exact call order.

## Problem 03 — Detect a broken vector comparison contract

**Challenge.** Reproduce the source lesson's design where vector equality compares coordinates but `<` compares Euclidean magnitude. Find two different vectors whose lengths match. Show how deriving `<=` as `a == b or a < b` yields neither `a <= b` nor `b <= a`. Explain why a magnitude-only **sort key** may be a better API.

### Worked solution

In [4]:
class MagnitudeVector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"MagnitudeVector({self.x}, {self.y})"

    def __eq__(self, other):
        if not isinstance(other, MagnitudeVector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __lt__(self, other):
        if not isinstance(other, MagnitudeVector):
            return NotImplemented
        return hypot(self.x, self.y) < hypot(other.x, other.y)

    def __le__(self, other):
        if not isinstance(other, MagnitudeVector):
            return NotImplemented
        return self == other or self < other


north = MagnitudeVector(0, 5)
east = MagnitudeVector(5, 0)
assert not (north == east)
assert hypot(north.x, north.y) == hypot(east.x, east.y) == 5
assert not (north < east) and not (east < north)
assert not (north <= east) and not (east <= north)

by_magnitude = sorted([north, east], key=lambda v: hypot(v.x, v.y))
assert by_magnitude == [north, east]  # Stable sort preserves equal-key input order.
print("Same magnitude; distinct coordinates; neither <= result is True.")

Same magnitude; distinct coordinates; neither <= result is True.


**Why this works.** `==` uses coordinates while ordering treats equal-length vectors as ties. This is a preorder-like ranking mixed with coordinate equality, not a coherent total order. Use `key=...` to make the desired ranking explicit; if the object itself must support a total order, add a deterministic tie-breaker.

**Further challenge (try before reading the next problem).** Construct three vectors of the same length and explain what `sorted` promises for equal key values.

## Problem 04 — Implement a coherent immutable 2-D vector

**Challenge.** Design a `Vector2D` with type/finite-number validation, informative `repr`, coordinate equality, magnitude-first **total ordering** with coordinate tie-breakers, `abs`, and a matching hash. Keep `bool` out of numeric inputs and decline cross-type comparisons.

### Worked solution

In [5]:
@total_ordering
class Vector2D:
    __slots__ = ("_x", "_y")

    def __init__(self, x: int | float, y: int | float):
        if any(type(value) not in (int, float) or not isfinite(value)
               for value in (x, y)):
            raise ValueError("coordinates must be finite int/float values (not bool)")
        object.__setattr__(self, "_x", x)
        object.__setattr__(self, "_y", y)

    def __setattr__(self, name, value):
        raise AttributeError("Vector2D instances are immutable")

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __repr__(self):
        return f"Vector2D(x={self.x!r}, y={self.y!r})"

    def __abs__(self):
        return hypot(self.x, self.y)

    def _order_key(self):
        return (abs(self), self.x, self.y)

    def __eq__(self, other):
        if type(other) is not Vector2D:
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __lt__(self, other):
        if type(other) is not Vector2D:
            return NotImplemented
        return self._order_key() < other._order_key()

    def __hash__(self):
        return hash((self.x, self.y))


vectors = [Vector2D(0, 5), Vector2D(5, 0), Vector2D(0, 0),
           Vector2D(3, 4), Vector2D(0.0, 5.0)]
ordered_vectors = sorted(vectors)
assert ordered_vectors[0] == Vector2D(0, 0)
assert ordered_vectors[-1] == Vector2D(5, 0)
assert Vector2D(3, 4) < Vector2D(5, 0)
assert Vector2D(0, 5) == Vector2D(0.0, 5.0)
assert hash(Vector2D(0, 5)) == hash(Vector2D(0.0, 5.0))
assert len({Vector2D(0, 5), Vector2D(0.0, 5.0)}) == 1
assert "immutable" in expect_raises(AttributeError, lambda: setattr(vectors[0], "_x", 99))
for invalid in [(True, 1), (float("nan"), 0), (0, float("inf"))]:
    expect_raises(ValueError, lambda pair=invalid: Vector2D(*pair))
assert Vector2D(1, 2).__eq__((1, 2)) is NotImplemented
print("Sorted vectors:", ordered_vectors)

Sorted vectors: [Vector2D(x=0, y=0), Vector2D(x=0, y=5), Vector2D(x=0.0, y=5.0), Vector2D(x=3, y=4), Vector2D(x=5, y=0)]


**Why this works.** Ties in magnitude are broken by coordinates, so an ordering tie now agrees with coordinate equality. The explicit hash follows equality rather than the magnitude. Immutability protects hash keys from state changes; validating non-finite values prevents NaN-related ordering anomalies. `hypot` is numerically preferable to `sqrt(x*x + y*y)` for many extreme magnitudes, though its result still has finite-precision limits.

**Further challenge (try before reading the next problem).** For exact integer-coordinate ordering over huge integers, compare squared magnitudes in integer arithmetic instead of relying on floating-point `hypot`.

## Problem 05 — Support tuple equality without breaking hash contracts

**Challenge.** A coordinate object should equal a `(x, y)` tuple in **both** operand orders. Validate tuple shape without unpacking arbitrary lengths; implement a hash compatible with equal tuples. Explain why cross-type equality should be added only when the API really needs it.

### Worked solution

In [6]:
class Coordinate:
    __slots__ = ("x", "y")

    def __init__(self, x: int, y: int):
        if type(x) is not int or type(y) is not int:
            raise TypeError("Coordinate requires exactly two integers")
        object.__setattr__(self, "x", x)
        object.__setattr__(self, "y", y)

    def __setattr__(self, name, value):
        raise AttributeError("Coordinate is immutable")

    def __repr__(self):
        return f"Coordinate({self.x}, {self.y})"

    def __eq__(self, other):
        if type(other) is Coordinate:
            return (self.x, self.y) == (other.x, other.y)
        if type(other) is tuple:
            return len(other) == 2 and (self.x, self.y) == other
        return NotImplemented

    def __hash__(self):
        return hash((self.x, self.y))


c = Coordinate(7, 9)
assert c == (7, 9) and (7, 9) == c
assert c != (7, 8) and c != (7, 9, 11)
assert c.__eq__([7, 9]) is NotImplemented
assert hash(c) == hash((7, 9))
assert len({c, (7, 9), Coordinate(7, 9)}) == 1
assert {c: "found"}[(7, 9)] == "found"
print("Cross-type equality is symmetric and hash-compatible.")

Cross-type equality is symmetric and hash-compatible.


**Why this works.** Returning `NotImplemented` lets a tuple's equality attempt fall through to `Coordinate.__eq__`. The hash uses the same coordinate tuple so values that compare equal have identical hashes. Supporting tuples expands the public equivalence relation and may affect set/dict keys; do not add it casually.

**Further challenge (try before reading the next problem).** Try the deliberately malformed tuples `()`, `(1,)`, and `(1, 2, 3)` and verify no exceptions escape.

## Problem 06 — Investigate reflected-method subclass precedence

**Challenge.** Demonstrate that if the right operand is a proper subclass and overrides the reflected method, Python may invoke the subclass's reflected method **before** the left operand's method. Assert the actual trace. This is why dispatch is not adequately described as a literal textual swap.

### Worked solution

In [7]:
dispatch_log = []

class Parent:
    def __lt__(self, other):
        dispatch_log.append("Parent.__lt__")
        return True

class Child(Parent):
    def __gt__(self, other):
        dispatch_log.append("Child.__gt__")
        return False


answer = Parent() < Child()
assert answer is False
assert dispatch_log == ["Child.__gt__"]
print("Subclass-priority trace:", dispatch_log)

dispatch_log.clear()
assert Parent() < Parent()
assert dispatch_log == ["Parent.__lt__"]

Subclass-priority trace: ['Child.__gt__']


**Why this works.** For a proper subclass on the right, Python gives an overridden reflected implementation priority. The subclass can therefore refine behavior without first waiting for the base class to return `NotImplemented`.

**Further challenge (try before reading the next problem).** Modify `Child.__gt__` to return `NotImplemented`; inspect the subsequent trace.

## Problem 07 — Model a true partial order

**Challenge.** Use mathematical set inclusion (`⊆` and `⊂`) to build a class with a **partial order**. Show an incomparable pair and why `@total_ordering` cannot turn incomparability into a meaningful total order. Provide a separate deterministic sort key for display purposes.

### Worked solution

In [8]:
class FeatureSet:
    def __init__(self, values):
        self.values = frozenset(values)

    def __repr__(self):
        return f"FeatureSet({sorted(self.values)!r})"

    def __eq__(self, other):
        if not isinstance(other, FeatureSet):
            return NotImplemented
        return self.values == other.values

    def __le__(self, other):
        if not isinstance(other, FeatureSet):
            return NotImplemented
        return self.values <= other.values

    def __lt__(self, other):
        if not isinstance(other, FeatureSet):
            return NotImplemented
        return self.values < other.values

    def __ge__(self, other):
        if not isinstance(other, FeatureSet):
            return NotImplemented
        return self.values >= other.values

    def __gt__(self, other):
        if not isinstance(other, FeatureSet):
            return NotImplemented
        return self.values > other.values

    def __hash__(self):
        return hash(self.values)


a = FeatureSet({"python"})
b = FeatureSet({"sql"})
c = FeatureSet({"python", "sql"})
assert a < c and b < c
assert not (a < b or b < a or a == b or a <= b or b <= a)
assert a <= a and c >= a
assert sorted([b, c, a], key=lambda obj: (len(obj.values), tuple(sorted(obj.values)))) == [a, b, c]
print("Incomparability is meaningful; display key is an independent policy.")

Incomparability is meaningful; display key is an independent policy.


**Why this works.** Subset inclusion is reflexive, antisymmetric and transitive but not total: disjoint singleton sets cannot be compared by inclusion. Auto-generating missing operators from one relation does not manufacture a mathematically justified ordering.

**Further challenge (try before reading the next problem).** Show three sets whose direct pairwise comparisons violate a proposed naive alphabetic display order.

## Problem 08 — Use `@total_ordering` on a version object

**Challenge.** Create a hashable `Version` with three nonnegative integer components. Implement exactly `__eq__` and `__lt__`, generate the remaining methods with `@total_ordering`, and verify every operator including unsupported operand behavior.

### Worked solution

In [9]:
@total_ordering
class Version:
    __slots__ = ("major", "minor", "patch")

    def __init__(self, major: int, minor: int, patch: int):
        components = (major, minor, patch)
        if any(type(v) is not int or v < 0 for v in components):
            raise ValueError("version components must be nonnegative integers")
        object.__setattr__(self, "major", major)
        object.__setattr__(self, "minor", minor)
        object.__setattr__(self, "patch", patch)

    def __setattr__(self, name, value):
        raise AttributeError("Version is immutable")

    def _key(self):
        return self.major, self.minor, self.patch

    def __repr__(self):
        return f"Version({self.major}, {self.minor}, {self.patch})"

    def __eq__(self, other):
        if type(other) is not Version:
            return NotImplemented
        return self._key() == other._key()

    def __lt__(self, other):
        if type(other) is not Version:
            return NotImplemented
        return self._key() < other._key()

    def __hash__(self):
        return hash(self._key())


v1, v2, v3 = Version(1, 9, 9), Version(2, 0, 0), Version(1, 9, 9)
assert v1 < v2 and v1 <= v2 and v2 > v1 and v2 >= v1
assert v1 == v3 and v1 <= v3 and v1 >= v3 and not (v1 != v3)
assert v1.__lt__("2.0.0") is NotImplemented
assert (v1 == "1.9.9") is False
expect_raises(TypeError, lambda: v1 < "2.0.0")
assert len({v1, v2, v3}) == 2
print("Six comparison operators work with two explicit implementations.")

Six comparison operators work with two explicit implementations.


**Why this works.** The decorator supplies ordering operations that are absent from the class; it does not override ones already defined. `__eq__` is inherited by every Python object, but explicitly implementing meaningful value equality is usually essential. A coherent comparison key makes generated operations reliable.

**Further challenge (try before reading the next problem).** Compare this class with `dataclasses.dataclass(order=True)` in the next problem.

## Problem 09 — Contrast `dataclass(order=True)` with manual comparisons

**Challenge.** Use `@dataclass(frozen=True, order=True)` to generate ordering from field declaration order. Show why a descriptive field marked `compare=False` should not affect identity/order, and show what happens when comparing a dataclass instance to a tuple.

### Worked solution

In [10]:
from dataclasses import field

@dataclass(frozen=True, order=True, slots=True)
class RankedRecord:
    priority: int
    timestamp: int
    label: str = field(compare=False)


x = RankedRecord(1, 10, "first label")
y = RankedRecord(1, 10, "another label")
z = RankedRecord(2, 0, "later priority")
assert x == y and hash(x) == hash(y)
assert x < z and x <= y and z > x
assert not (x == (1, 10))
expect_raises(TypeError, lambda: x < (1, 10))
assert sorted([z, y, x]) == [y, x, z]
print("Dataclass ordering follows comparison-enabled fields in declaration order.")

Dataclass ordering follows comparison-enabled fields in declaration order.


**Why this works.** Dataclasses generate comparisons for the same class using selected fields as a tuple. `compare=False` excludes a field from equality and ordering and, by default, from generated hashing decisions tied to comparison. `frozen=True` is shallow immutability; avoid storing mutable, equality-relevant structures inside it.

**Further challenge (try before reading the next problem).** Reorder `priority` and `timestamp` in the declaration and predict how sorting changes.

## Problem 10 — Reject NaN and infinite values at a comparison boundary

**Challenge.** Explain why ordinary floating-point NaN violates familiar trichotomy assumptions; implement an ordered `Score` that rejects `NaN`, infinities and boolean inputs. Confirm negative zero is equal to positive zero with equal hashes.

### Worked solution

In [11]:
nan = float("nan")
assert nan != nan
assert not (nan < 3.0) and not (nan > 3.0) and not (nan == 3.0)

@total_ordering
@dataclass(frozen=True, slots=True, eq=False)
class Score:
    value: float

    def __post_init__(self):
        if type(self.value) not in (int, float) or not isfinite(self.value):
            raise ValueError("Score needs a finite real-valued int/float, not bool")

    def __eq__(self, other):
        if type(other) is not Score:
            return NotImplemented
        return self.value == other.value

    def __lt__(self, other):
        if type(other) is not Score:
            return NotImplemented
        return self.value < other.value

    def __hash__(self):
        return hash(self.value)


assert Score(-0.0) == Score(0.0)
assert hash(Score(-0.0)) == hash(Score(0.0))
assert Score(2.0) < Score(3.0)
for bad in (nan, float("inf"), float("-inf"), True, "1"):
    expect_raises(ValueError, lambda value=bad: Score(value))
assert (Score(1) == object()) is False
expect_raises(TypeError, lambda: Score(1) < object())
print("Non-finite and ambiguous inputs were rejected.")

Non-finite and ambiguous inputs were rejected.


**Why this works.** NaN compares unequal even to itself, and all ordinary less-than/greater-than comparisons with NaN are false. Boundary validation allows the class to promise a conventional order; signed zeros must share hashes because they compare equal.

**Further challenge (try before reading the next problem).** Explore whether a domain with arbitrary `Decimal` values needs extra handling for decimal NaNs and signaling NaNs.

## Problem 11 — Design stable multi-key sorting without overloads

**Challenge.** You are given immutable tickets with severity (higher means more urgent), arrival index (lower means earlier) and label. Sort by descending severity then ascending arrival. Explain why `sorted(key=...)` is clearer than defining a questionable global ordering on `Ticket`. Demonstrate stability for equal keys.

### Worked solution

In [12]:
@dataclass(frozen=True, slots=True)
class Ticket:
    severity: int
    arrival: int
    label: str


tickets = [Ticket(2, 4, "A"), Ticket(5, 3, "B"), Ticket(5, 1, "C"),
           Ticket(2, 4, "D"), Ticket(5, 1, "E")]
queue = sorted(tickets, key=lambda t: (-t.severity, t.arrival))
assert [t.label for t in queue] == ["C", "E", "B", "A", "D"]
assert [t.label for t in sorted(tickets, key=lambda t: t.arrival)] == ["C", "E", "B", "A", "D"]
assert [t.label for t in sorted(tickets, key=lambda t: t.severity, reverse=True)] == ["B", "C", "E", "A", "D"]
print("Priority queue (stable ties):", [t.label for t in queue])

Priority queue (stable ties): ['C', 'E', 'B', 'A', 'D']


**Why this works.** Python sorting is stable: records with equal keys keep their original relative order. A tuple key documents the ranking policy at its use site; negating numeric severity avoids `reverse=True` reversing secondary tie-breakers as well.

**Further challenge (try before reading the next problem).** Change label or arrival independently and inspect which permutations remain stable.

## Problem 12 — Build a robust `heapq` priority queue

**Challenge.** A heap of `(priority, payload)` tuples can try to compare two payloads when priorities tie. Reproduce the failure and fix it with a monotonically increasing insertion counter. Verify deterministic FIFO behavior on equal priorities.

### Worked solution

In [13]:
@dataclass(frozen=True, slots=True)
class Job:
    name: str


bad_heap = [(1, Job("alpha"))]
expect_raises(TypeError, lambda: heapq.heappush(bad_heap, (1, Job("beta"))))

serial = count()
good_heap = []
for priority, name in [(2, "ordinary"), (1, "first"), (1, "second"), (3, "later")]:
    heapq.heappush(good_heap, (priority, next(serial), Job(name)))

popped = [heapq.heappop(good_heap)[2].name for _ in range(4)]
assert popped == ["first", "second", "ordinary", "later"]
print("Stable heap order:", popped)

Stable heap order: ['first', 'second', 'ordinary', 'later']


**Why this works.** `heapq` compares heap entries with `<`, which for tuples proceeds to later components only when earlier components tie. A unique integer counter makes payload comparisons unnecessary; its incrementing values encode FIFO ordering on ties.

**Further challenge (try before reading the next problem).** Make the priority negative to build a max-priority heap without ever comparing payloads.

## Problem 13 — Audit equality and hashing for mutable objects

**Challenge.** Show that a mutable `@dataclass(eq=True)` normally disables hashing; then use a frozen dataclass as a safe dictionary key. Illustrate, in an isolated example, why changing the state of an object after inserting it into a set would violate lookup expectations.

### Worked solution

In [14]:
@dataclass
class MutablePoint:
    x: int
    y: int

@dataclass(frozen=True, slots=True)
class FrozenPoint:
    x: int
    y: int


m = MutablePoint(1, 2)
assert MutablePoint.__hash__ is None
expect_raises(TypeError, lambda: hash(m))
f1, f2 = FrozenPoint(1, 2), FrozenPoint(1, 2)
assert f1 == f2 and hash(f1) == hash(f2)
assert {f1: "here"}[f2] == "here"
expect_raises(Exception, lambda: setattr(f1, "x", 100))

# Controlled demonstration: NEVER do this with real dictionary/set keys.
class UnsafeMutableKey:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, UnsafeMutableKey) and self.value == other.value

    def __hash__(self):
        return hash(self.value)

key = UnsafeMutableKey(10)
original_hash = hash(key)
key.value = 11
assert hash(key) != original_hash
print("Mutating an inserted hash key could strand it in the wrong hash bucket.")

Mutating an inserted hash key could strand it in the wrong hash bucket.


**Why this works.** `a == b` implies `hash(a) == hash(b)` whenever both are hashable; this is necessary, not sufficient, for equality. Dataclass defaults protect mutable value objects by making them unhashable, while `frozen=True` enables a generated value hash when fields are hashable. Avoid `unsafe_hash=True` for objects whose comparison-relevant fields can change.

**Further challenge (try before reading the next problem).** Explain why hash collisions are allowed even when two values are not equal.

## Problem 14 — Count comparisons and choose an appropriate sorting API

**Challenge.** Compare the number of calls for `sorted(objects)` (rich comparisons) versus `sorted(objects, key=...)`. Make the experiment deterministic and verify that a key function is evaluated once per input element, without claiming a universal comparison count.

### Worked solution

In [15]:
comparison_calls = 0
key_calls = 0

@total_ordering
class CountingRecord:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if type(other) is not CountingRecord:
            return NotImplemented
        return self.value == other.value

    def __lt__(self, other):
        global comparison_calls
        if type(other) is not CountingRecord:
            return NotImplemented
        comparison_calls += 1
        return self.value < other.value


def record_key(record):
    global key_calls
    key_calls += 1
    return record.value


items = [CountingRecord(n) for n in (4, 7, 2, 8, 1, 5, 3, 6)]
by_comparison = sorted(items)
by_key = sorted(items, key=record_key)
assert [x.value for x in by_comparison] == list(range(1, 9))
assert [x.value for x in by_key] == list(range(1, 9))
assert key_calls == len(items)
assert comparison_calls > 0
print(f"Elements: {len(items)}; rich < calls: {comparison_calls}; key calls: {key_calls}")

Elements: 8; rich < calls: 19; key calls: 8


**Why this works.** The sorting API caches key values rather than recomputing the Python key function for every element comparison. Counts of rich comparisons depend on input order and the sorting algorithm; the point is clarity and cached key extraction, not a blanket speed guarantee.

**Further challenge (try before reading the next problem).** Build an expensive pure key function and measure work with counters instead of noisy wall-clock timing.

## Problem 15 — Adapt a legacy three-way comparator safely

**Challenge.** Convert a legacy integer comparison function returning negative/zero/positive into a key using `cmp_to_key`. Verify it handles ties and agrees with the direct key-based sort. Explain why a modern key function should usually be preferred when feasible.

### Worked solution

In [16]:
@dataclass(frozen=True)
class Package:
    weight: int
    name: str


def legacy_compare(a: Package, b: Package) -> int:
    ka = (a.weight, a.name)
    kb = (b.weight, b.name)
    return (ka > kb) - (ka < kb)


packages = [Package(8, "z"), Package(3, "b"), Package(3, "a"), Package(8, "z")]
via_adapter = sorted(packages, key=cmp_to_key(legacy_compare))
via_key = sorted(packages, key=lambda p: (p.weight, p.name))
assert via_adapter == via_key
assert legacy_compare(packages[0], packages[-1]) == 0
assert legacy_compare(packages[1], packages[2]) > 0
assert legacy_compare(packages[2], packages[1]) < 0
print("Comparator adapter and tuple-key sorting agree:", via_adapter)

Comparator adapter and tuple-key sorting agree: [Package(weight=3, name='a'), Package(weight=3, name='b'), Package(weight=8, name='z'), Package(weight=8, name='z')]


**Why this works.** A correctly designed three-way comparator returns 0 for equivalent ordering keys and opposite signs for swapped operands. `cmp_to_key` is useful for interoperability, but tuple-key sorting makes the policy more visible and extracts the key once per element.

**Further challenge (try before reading the next problem).** Write an inconsistent comparator (rock–paper–scissors) and explain why no sorting algorithm can make it a total order.

## Problem 16 — Exhaustively test ordering laws on a small domain

**Challenge.** Write a reusable invariant checker for an ordered, hashable type. Test reflexivity, symmetry, trichotomy, equivalence of equality with absence of `<` in either direction, transitivity of `<`, and equal-hash consistency on a finite sample. Use it against both `Version` and `Vector2D`.

### Worked solution

In [17]:
def check_total_order_laws(samples):
    """Check representative laws; passing a finite sample is evidence, not a proof."""
    for a in samples:
        assert a == a
        assert a <= a and a >= a
        assert not (a < a) and not (a > a)
    for a, b in product(samples, repeat=2):
        assert (a == b) == (b == a)
        assert int(a < b) + int(a == b) + int(a > b) == 1
        assert (a <= b) == (a < b or a == b)
        assert (a >= b) == (a > b or a == b)
        assert (a != b) == (not (a == b))
        if a == b:
            assert hash(a) == hash(b)
    for a, b, c in product(samples, repeat=3):
        if a < b and b < c:
            assert a < c
        if a == b and b == c:
            assert a == c
    return len(samples), len(samples) ** 3


version_laws = check_total_order_laws(
    [Version(a, b, 0) for a, b in product(range(2), repeat=2)] + [Version(1, 0, 0)]
)
vector_laws = check_total_order_laws(
    [Vector2D(x, y) for x, y in product((-1, 0, 1), repeat=2)]
)
assert version_laws[1] == 125
assert vector_laws[1] == 729
print("Finite-domain law checks (items, triples):", version_laws, vector_laws)

Finite-domain law checks (items, triples): (5, 125) (9, 729)


**Why this works.** Tests of semantic invariants catch failures that individual examples may miss, especially around ordering ties and equal hashes. A finite enumeration does not prove the laws for all possible inputs; use it alongside a clear documented contract and (in larger projects) property-based testing.

**Further challenge (try before reading the next problem).** Add checks for partial-order reflexivity, antisymmetry and transitivity, but deliberately omit trichotomy.

## Problem 17 — Compare money only within one currency

**Challenge.** Implement immutable monetary amounts using `Decimal`. Amounts in the same currency may be ordered; equality across currencies is `False`, while ordering different currencies must raise a **domain-specific `ValueError`** (not silently assume an exchange rate). Unsupported Python operand types should still return `NotImplemented`.

### Worked solution

In [18]:
@total_ordering
@dataclass(frozen=True, slots=True, eq=False)
class Money:
    amount: Decimal
    currency: str

    def __post_init__(self):
        if type(self.amount) is not Decimal or not self.amount.is_finite():
            raise ValueError("amount must be a finite Decimal")
        if (type(self.currency) is not str or len(self.currency) != 3
                or not self.currency.isascii() or not self.currency.isalpha()
                or self.currency != self.currency.upper()):
            raise ValueError("currency must be three uppercase ASCII letters")

    def __eq__(self, other):
        if type(other) is not Money:
            return NotImplemented
        return (self.currency, self.amount) == (other.currency, other.amount)

    def __lt__(self, other):
        if type(other) is not Money:
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("cannot order money with different currencies")
        return self.amount < other.amount

    def __hash__(self):
        return hash((self.currency, self.amount))


usd5 = Money(Decimal("5.00"), "USD")
usd7 = Money(Decimal("7"), "USD")
euro5 = Money(Decimal("5"), "EUR")
assert usd5 < usd7 and usd5 <= usd7 and usd7 > usd5
assert usd5 != euro5 and (usd5 == euro5) is False
assert usd5 == Money(Decimal("5.0"), "USD")
assert hash(usd5) == hash(Money(Decimal("5.0"), "USD"))
assert "different currencies" in expect_raises(ValueError, lambda: usd5 < euro5)
assert "different currencies" in expect_raises(ValueError, lambda: usd5 >= euro5)
assert usd5.__lt__(5) is NotImplemented
expect_raises(TypeError, lambda: usd5 < 5)
for invalid in [(Decimal("NaN"), "USD"), (Decimal("5"), "usd"), (5.0, "USD")]:
    expect_raises(ValueError, lambda args=invalid: Money(*args))
print("Money contract: comparable within a currency, explicit error otherwise.")

Money contract: comparable within a currency, explicit error otherwise.


**Why this works.** Different currencies represent valid `Money` operands but have no intrinsic exchange-rate-independent order, so a deliberate domain error is appropriate. Non-`Money` operands are an *unsupported Python type* and should return `NotImplemented` for proper dispatcher fallback. This is a domain-specific comparison relation, **not** a global total order across all `Money` instances.

**Further challenge (try before reading the next problem).** Implement explicit conversion as a separate function taking a specified exchange rate and rounding policy.

## Problem 18 — Capstone: semantic-version precedence and build metadata

**Challenge.** Implement a compact SemVer 2.0 precedence model supporting major/minor/patch, prerelease dot-identifiers (numeric vs alphanumeric), and optional build metadata. Validate leading zeros for numeric version/pre-release identifiers, reject malformed strings, treat build metadata as **irrelevant to precedence**, and choose an explicitly documented equality/hash contract that is consistent with the order. Verify the official example precedence chain.

### Worked solution

In [19]:
SEMVER_PATTERN = re.compile(
    r"^(0|[1-9][0-9]*)\.(0|[1-9][0-9]*)\.(0|[1-9][0-9]*)"
    r"(?:-([0-9A-Za-z-]+(?:\.[0-9A-Za-z-]+)*))?"
    r"(?:\+([0-9A-Za-z-]+(?:\.[0-9A-Za-z-]+)*))?$"
)

@total_ordering
@dataclass(frozen=True, slots=True, eq=False)
class SemVer:
    major: int
    minor: int
    patch: int
    prerelease: tuple[str, ...] = ()
    build: tuple[str, ...] = ()

    def __post_init__(self):
        if any(type(n) is not int or n < 0
               for n in (self.major, self.minor, self.patch)):
            raise ValueError("major/minor/patch must be nonnegative integers")
        for part in self.prerelease + self.build:
            if type(part) is not str or not re.fullmatch(r"[0-9A-Za-z-]+", part):
                raise ValueError("invalid SemVer identifier")
        for part in self.prerelease:
            if part.isascii() and part.isdecimal() and len(part) > 1 and part[0] == "0":
                raise ValueError("numeric prerelease identifier has leading zero")

    @classmethod
    def parse(cls, text: str) -> SemVer:
        if type(text) is not str:
            raise TypeError("version text must be a string")
        match = SEMVER_PATTERN.fullmatch(text)
        if match is None:
            raise ValueError(f"invalid semantic version: {text!r}")
        major, minor, patch, prerelease, build = match.groups()
        return cls(int(major), int(minor), int(patch),
                   tuple(prerelease.split(".")) if prerelease else (),
                   tuple(build.split(".")) if build else ())

    def _precedence(self):
        numeric = (self.major, self.minor, self.patch)
        if not self.prerelease:
            return numeric, (1,)  # A release is later than any prerelease.
        identifiers = tuple(
            (0, int(part)) if part.isdecimal() else (1, part)
            for part in self.prerelease
        )
        return numeric, (0, identifiers)

    def __eq__(self, other):
        if type(other) is not SemVer:
            return NotImplemented
        return self._precedence() == other._precedence()

    def __lt__(self, other):
        if type(other) is not SemVer:
            return NotImplemented
        return self._precedence() < other._precedence()

    def __hash__(self):
        return hash(self._precedence())

    def __str__(self):
        core = f"{self.major}.{self.minor}.{self.patch}"
        pre = "-" + ".".join(self.prerelease) if self.prerelease else ""
        build = "+" + ".".join(self.build) if self.build else ""
        return core + pre + build


chain = [
    "1.0.0-alpha", "1.0.0-alpha.1", "1.0.0-alpha.beta",
    "1.0.0-beta", "1.0.0-beta.2", "1.0.0-beta.11",
    "1.0.0-rc.1", "1.0.0",
]
parsed = [SemVer.parse(item) for item in chain]
assert sorted(reversed(parsed)) == parsed
assert all(a < b for a, b in zip(parsed, parsed[1:]))
assert SemVer.parse("1.0.0+sha.123") == SemVer.parse("1.0.0+sha.456")
assert hash(SemVer.parse("1.0.0+sha.123")) == hash(SemVer.parse("1.0.0+sha.456"))
assert len({SemVer.parse("1.0.0+foo"), SemVer.parse("1.0.0+bar")}) == 1
assert SemVer.parse("2.1.0-5") < SemVer.parse("2.1.0-alpha")
assert SemVer.parse("1.0.0-alpha") < SemVer.parse("1.0.0-alpha.1")
assert str(SemVer.parse("2.0.1-beta.3+linux.7")) == "2.0.1-beta.3+linux.7"
for invalid in ("01.0.0", "1.02.0", "1.0.00", "1.0.0-alpha.01", "1.0.0-",
                "1.0.0+", "v1.0.0", "1.0.0-α", "1.0", "1.0.0.1"):
    expect_raises(ValueError, lambda item=invalid: SemVer.parse(item))
expect_raises(TypeError, lambda: SemVer.parse(123))
expect_raises(ValueError, lambda: SemVer(1, 0, 0, ("01",)))
print("SemVer precedence chain:", " < ".join(chain))
print("Distinct build strings compare equal *by this model's explicit policy*.")

SemVer precedence chain: 1.0.0-alpha < 1.0.0-alpha.1 < 1.0.0-alpha.beta < 1.0.0-beta < 1.0.0-beta.2 < 1.0.0-beta.11 < 1.0.0-rc.1 < 1.0.0
Distinct build strings compare equal *by this model's explicit policy*.


**Why this works.** The comparison key puts numeric identifiers before alphanumeric identifiers, compares dot-parts lexicographically, makes shorter identical-prefix prereleases earlier, and places the final release after prereleases. SemVer 2.0 says build metadata does not affect **precedence**; this educational class intentionally also ignores build metadata for **equality and hashing** to obtain a coherent total ordering. Real tools may instead preserve full version identity and implement precedence as a separate comparison/key; choose and document the contract that fits your application.

**Further challenge (try before reading the next problem).** Extend parsing to support strict maximum input lengths or precedence-only keys for a package manager that distinguishes different builds.

---
## Final integrated regression run

The checks below repeat central boundary conditions so this notebook acts as a single executable exercise set. A successful run prints **ALL CHECKS PASSED**. If you change a solution, rerun the notebook from a fresh kernel to catch accidental state dependencies.

In [20]:
assert Vector2D(0, 5) < Vector2D(5, 0)
assert (7, 9) == Coordinate(7, 9)
assert Version(1, 2, 3) < Version(1, 2, 4)
assert Money(Decimal("2.0"), "USD") == Money(Decimal("2.00"), "USD")
assert SemVer.parse("1.0.0-rc.1") < SemVer.parse("1.0.0")
assert check_total_order_laws([Version(0, 0, 0), Version(1, 0, 0), Version(1, 0, 0)]) == (3, 27)
print("ALL CHECKS PASSED — 18 problems with runnable, assertion-tested solutions.")

ALL CHECKS PASSED — 18 problems with runnable, assertion-tested solutions.


## Practical review checklist

1. Does equality represent precisely the intended value identity? What happens across types and subclasses?
2. Is ordering meaningful for *every* pair of supported objects, or is the domain only partially ordered?
3. Does an equality tie agree with an ordering tie? Are tie-breakers and stable sorting deliberate?
4. Are `NotImplemented`, `False`, `TypeError`, and domain-specific errors used for different purposes?
5. Do equal hashable objects have equal hashes, and are their relevant fields immutable?
6. Have non-finite numbers, shape mismatches, numeric type mixing, and invalid inputs been tested?
7. Would a `key=` callable be simpler than adding comparison operators to the class?
8. Can the tests be run from a freshly restarted kernel, with no external dependencies?

**Further independent projects:** design a `Rational` normalized with `math.gcd`, order intervals by containment (partial order), implement natural filename sorting through `key=`, and verify a 3-way legacy comparator's antisymmetry/transitivity before adapting it.